In [1]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/ammara-asif/fyp.git"   # <-- EDIT ONCE
BRANCH = "nested-b-d"   # the branch with the code this notebook needs (set back to "main" after merging)

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", url, CODE], check=True)
# fetch + checkout + pull, so a clone left on another branch earlier in the session switches over
subprocess.run(["git", "-C", CODE, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", CODE, "pull", "-q", "origin", BRANCH], check=True)
print(BRANCH, subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
_MINE = {"fyp_infra", "stage1_runner", "stage1_analysis", "stage1_drift_diagnostic",
         "stage1_structural", "nested_builder"}
for _m in list(sys.modules):
    if _m in _MINE or _m == "src" or _m.startswith("src."):   # src.* = the team's nested code
        sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop for training: a CPU run is ~12x slower and its timings are useless for budgeting.
REQUIRE_GPU = False   # analysis only: CPU runtime is fine
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

Mounted at /content/drive
nested-b-d fed9369 nested variants



/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ammara-asif125 (kashaf-saleem09-institute-of-business-administration) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


{'root': '/content/drive/MyDrive/fyp_nested_cl', 'data': '/content/drive/MyDrive/fyp_nested_cl/data', 'results': '/content/drive/MyDrive/fyp_nested_cl/results', 'logs': '/content/drive/MyDrive/fyp_nested_cl/logs', 'checkpoints': '/content/drive/MyDrive/fyp_nested_cl/checkpoints'}
GPU: Tesla T4


In [11]:
import stage1_analysis as sa
import stage1_structural as st
STAGES = ["stage1_control", "stage1_nested_control"]
TRAIN_RUNS = sa.list_runs(paths, stage=STAGES)
for r in TRAIN_RUNS:
    c = r["config"]; print(r["run_id"], sr.arm_label(c), "lr", c["lr"], "seed", c["seed"])
SCFGS = st.structural_configs(TRAIN_RUNS)

07a97dd12ec3 flat_nl lr 0.1 seed 0
1803d732791d llrd_fast_deep lr 0.6 seed 1
32e5df8b8886 flat lr 0.3 seed 2
48523f3dfeea nested_b lr 0.6 seed 1
4b2bd0082f5c flat lr 0.1 seed 0
78e133183087 flat_nl lr 0.1 seed 1
7b3073002436 flat lr 0.3 seed 0
892486433e6d flat lr 0.1 seed 1
9d5d9fcc4194 llrd_fast_deep lr 0.6 seed 0
b09e3cc29bda flat_nl lr 0.1 seed 2
b2316f2cb5f4 nested_d_lam0.01 lr 0.7 seed 0
b409e8563692 llrd_fast_deep lr 0.6 seed 2
badf7f571542 nested_b lr 0.6 seed 2
bf6b7bf5e853 llrd_slow_deep lr 0.5 seed 2
bf7c7b91a468 llrd_slow_deep lr 0.5 seed 1
c4b3004f3b61 nested_d_lam0.01 lr 0.7 seed 2
e3be0b8051fd nested_d_lam0.01 lr 0.7 seed 1
e3f9264b4607 flat lr 0.1 seed 2
ea0983312db5 nested_b lr 0.6 seed 0
f686172d0a03 flat lr 0.3 seed 1
f7af3b7e79aa llrd_slow_deep lr 0.5 seed 0


In [12]:
out = run_sweep(SCFGS, st.structural_run, paths)

[1/21] 1f493c365153 running
  checkpoint 1/20
  checkpoint 2/20
  checkpoint 3/20
  checkpoint 4/20
  checkpoint 5/20
  checkpoint 6/20
  checkpoint 7/20
  checkpoint 8/20
  checkpoint 9/20
  checkpoint 10/20
  checkpoint 11/20
  checkpoint 12/20
  checkpoint 13/20
  checkpoint 14/20
  checkpoint 15/20
  checkpoint 16/20
  checkpoint 17/20
  checkpoint 18/20
  checkpoint 19/20
  checkpoint 20/20
[2/21] b7c5dedbcd65 running
  checkpoint 1/20
  checkpoint 2/20
  checkpoint 3/20
  checkpoint 4/20
  checkpoint 5/20
  checkpoint 6/20
  checkpoint 7/20
  checkpoint 8/20
  checkpoint 9/20
  checkpoint 10/20
  checkpoint 11/20
  checkpoint 12/20
  checkpoint 13/20
  checkpoint 14/20
  checkpoint 15/20
  checkpoint 16/20
  checkpoint 17/20
  checkpoint 18/20
  checkpoint 19/20
  checkpoint 20/20
[3/21] ccacb1ec8e62 running
  checkpoint 1/20
  checkpoint 2/20
  checkpoint 3/20
  checkpoint 4/20
  checkpoint 5/20
  checkpoint 6/20
  checkpoint 7/20
  checkpoint 8/20
  checkpoint 9/20
  checkpoint

In [13]:
import numpy as np, pandas as pd
rows = []
for a in SCFGS:
    r = load_result(a, paths)
    if r is None: continue
    m = r["metrics"]; c = m["train_config"]
    row = {"arm": sr.arm_label(c), "lr": c["lr"], "seed": c["seed"]}
    for l, name in enumerate(m["level_modules"]):
        row[f"cka_to_final/{name}"] = m["cka_to_final_avg"][l]
        row[f"cka_consecutive/{name}"] = m["cka_consecutive_avg"][l]
    # group names differ between the two model classes (conv1+bn1+layer1 vs stem+layer1), so label by position
    n_g = len(m["grad_cos_avg"])
    for g, v in enumerate(m["grad_cos_avg"]):
        row["grad_cos/" + ("head" if g == n_g - 1 else f"group{g}")] = v
    rows.append(row)
sdf = pd.DataFrame(rows)
display(sdf.drop(columns="seed").groupby(["arm", "lr"]).agg(["mean", "std"]).T.round(3))

arm                           flat        flat_nl llrd_fast_deep  \
lr                             0.1    0.3     0.1            0.6   
cka_to_final/layer1    mean  0.748  0.745   0.757          0.730   
                       std   0.038  0.032   0.061          0.052   
cka_consecutive/layer1 mean  0.793  0.765   0.796          0.833   
                       std   0.030  0.021   0.013          0.021   
cka_to_final/layer3    mean  0.589  0.614   0.579          0.565   
                       std   0.054  0.027   0.040          0.058   
cka_consecutive/layer3 mean  0.555  0.584   0.548          0.578   
                       std   0.020  0.012   0.013          0.015   
cka_to_final/layer4    mean  0.407  0.387   0.402          0.388   
                       std   0.036  0.011   0.041          0.049   
cka_consecutive/layer4 mean  0.290  0.278   0.278          0.274   
                       std   0.020  0.029   0.029          0.027   
grad_cos/group0        mean  0.398  0.421   0.406          0.368   
                       std   0.113  0.031   0.119          0.059   
grad_cos/group1        mean  0.388  0.434   0.384          0.362   
                       std   0.101  0.057   0.094          0.074   
grad_cos/group2        mean  0.265  0.316   0.253          0.268   
                       std   0.077  0.061   0.070          0.048   
grad_cos/head          mean  0.207  0.258   0.203          0.203   
                       std   0.036  0.053   0.043          0.032   

arm                         llrd_slow_deep nested_b nested_d_lam0.01  
lr                                     0.5      0.6              0.7  
cka_to_final/layer1    mean          0.739    0.777            0.685  
                       std           0.061    0.012            0.051  
cka_consecutive/layer1 mean          0.798    0.826            0.796  
                       std           0.016    0.019            0.014  
cka_to_final/layer3    mean          0.537    0.607            0.475  
                       std           0.074    0.042            0.113  
cka_consecutive/layer3 mean          0.549    0.582            0.613  
                       std           0.027    0.024            0.040  
cka_to_final/layer4    mean          0.392    0.394            0.382  
                       std           0.045    0.050            0.037  
cka_consecutive/layer4 mean          0.317    0.277            0.296  
                       std           0.026    0.023            0.041  
grad_cos/group0        mean          0.330    0.427            0.337  
                       std           0.150    0.151            0.032  
grad_cos/group1        mean          0.329    0.395            0.331  
                       std           0.126    0.134            0.040  
grad_cos/group2        mean          0.236    0.291            0.202  
                       std           0.090    0.071            0.029  
grad_cos/head          mean          0.203    0.230            0.164  
                       std           0.069    0.049            0.037